# 03 — LPM / IM subclustering

**Feeds:** ED Fig 7c

**Position in the chain:** run the numbered stages in order

Ported from the original analysis. Saved cell outputs are from the original run, and paths in them appear as `<analysis-root>/...`.

**Changes from the original notebook**, so that it runs from this repository:
1. Paths. The first code cell finds the repository from any working directory inside it; the original looked for `src/` at most two levels up. `PROJECT_ROOT`, which was the analysis directory, is now `scrnaseq/chain_inputs/`, the shipped files that no notebook writes, in the same layout. `DEV_ROOT`, under which the notebook writes, is `$SCRNASEQ_RESULTS_ROOT/trunk_main_dev/` (default `scrnaseq/output/trunk_main_dev/`) instead of the analysis directory's `trunk_main_dev/`.
2. The SMD z-score files are read from `scrnaseq/chain_inputs/trunk_main_dev/results/smd_runs/` (`PROJECT_ROOT / "trunk_main_dev" / "results"`) instead of `RESULTS_DIR`, where they sat in the original analysis directory.
3. Removed the `.to_clipboard()` call, which copied a table to the system clipboard for pasting into the Supplementary Data spreadsheet and fail on a machine without a clipboard. The expression before each call is unchanged.
4. Added `EXPORT_DPI = 300` next to `NOTEBOOK_DPI`. The original notebook uses `EXPORT_DPI` in three later cells but never defines it, so it stopped with a `NameError` in a fresh kernel. 300 is the value `02_trunk_main` defines.
5. Removed the check that compared this notebook's Leiden labels (resolution 1.10, seed 5) with the same run's labels saved by a separate clustering-stability analysis, which is not part of this repository, and the line printing the agreement. The Leiden call itself is unchanged.

No other line of code was changed.


# 03 - LPM Subclustering

Depends on: 01_preprocessing, 02_trunk_main artifacts.

In [ ]:
import sys
from pathlib import Path

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "src" / "trunk_morph_ref").exists():
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError("Could not locate the repository root containing src/trunk_morph_ref/")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.trunk_morph_ref.paths import chain_inputs_root

# Inputs that no notebook writes: scrnaseq/chain_inputs/, laid out as the original analysis directory.
PROJECT_ROOT = chain_inputs_root(REPO_ROOT)

import os

os.chdir(REPO_ROOT)

## Setup and Imports


In [ ]:
import matplotlib

matplotlib.rcParams["pdf.fonttype"] = 42
matplotlib.rcParams["ps.fonttype"] = 42

NOTEBOOK_DPI = 110
EXPORT_DPI = 300


In [ ]:
import pickle
import random

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy
import seaborn as sns
from cmcrameri.cm import batlow
from scipy.sparse import coo_matrix, csr_matrix, load_npz, save_npz

In [ ]:
plt.rcParams["svg.fonttype"] = "none"  # Keep fonts editable in Illustrator

In [ ]:
from src.trunk_morph_ref.cell_ordering import ordered_cells_by_cluster_clustermap
from src.trunk_morph_ref.paths import init_output_paths, scrnaseq_results_root

# Outputs: $SCRNASEQ_RESULTS_ROOT (default scrnaseq/output/), under trunk_main_dev/ as in the original.
DEV_ROOT = scrnaseq_results_root(REPO_ROOT) / "trunk_main_dev"
RESULTS_DIR, MANUSCRIPT_FIG_DIR, EXTENDED_FIG_DIR = init_output_paths(DEV_ROOT)
from src.trunk_morph_ref.aggregation import cluster_averages_sparse_safe
from src.trunk_morph_ref.correlation import gene_corrcoef_sparse_safe
from src.trunk_morph_ref.plotting import (
    bold_selected_heatmap_yticklabels,
    hide_heatmap_axes_and_colorbar,
    keep_only_selected_heatmap_yticklabels,
    plot_empty_heatmap_axes,
)
from src.trunk_morph_ref.preprocessing import (
    assert_gene_id_index,
    build_legacy_smd_score_table,
    contains_symbol,
    ensure_gene_id_index,
)
from src.trunk_morph_ref.preprocessing import filter_genes_sparse_safe as filter_genes
from src.trunk_morph_ref.preprocessing import (
    install_scanpy_symbol_defaults,
    map_legacy_smd_scores_to_adata,
)
from src.trunk_morph_ref.preprocessing import (
    normalize_unit_variance_sparse_safe as normalize_unit_variance,
)
from src.trunk_morph_ref.preprocessing import (
    resolve_symbol_dict,
    resolve_symbols,
    symbols_missing,
    symbols_present,
    var_names_to_symbols,
)

# Use gene symbols for plot labels while keeping `var_names` as stable gene IDs.
install_scanpy_symbol_defaults(sc)

In [ ]:
from src.trunk_morph_ref.pipeline_io import (
    load_h5ad,
    load_npy,
    load_pickle,
    save_h5ad,
    save_json,
    save_npy,
    save_pickle,
    stage_dir,
)

In [ ]:
pre_path = stage_dir(RESULTS_DIR, "01_preprocessing")
trunk_path = stage_dir(RESULTS_DIR, "02_trunk_main")
adata_morph_raw = load_h5ad(pre_path / "adata_morph_raw.h5ad")
adata_morph_with_clusters = load_h5ad(trunk_path / "adata_morph_with_clusters.h5ad")
for _adata in [
    adata_morph_raw,
    adata_morph_with_clusters,
]:
    ensure_gene_id_index(_adata)
    assert_gene_id_index(_adata)

print("Loaded 01_preprocessing and 02_trunk_main intermediates")

## LPM Subclustering Analysis
Subclustering resolves anterior LPM, posterior LPM, and endothelial identities.


Pre-processing block for the combined LPM / IM / Endothelial subset before SMD.


In [ ]:
genes_cellcycle = [
    "MKI67",
    "CENPE",
    "SGO2",
    "KIF14",
    "PIF1",
    "NDC80",
    "CDCA8",
    "PLK1",
    "AURKA",
    "UBE2C",
    "ASPM",
    "TOP2A",
    "TPX2",
    "NUSAP1",
    "CDC20",
    "CKS2",
    "KPNA2",
    "TUBB4B",
    "DLGAP5",
    "BIRC5",
    "HMMR",
    "CCNB1",
    "ARL6IP1",
    "PTTG1",
    "UBE2S",
    "DUT",
    "HELLS",
    "CLSPN",
    "RRM2",
    "PCLAF",
    "TYMS",
    "KIF18B",
    "SMC4",
    "HIST1H4C",
    "DIAPH3",
    "RFC3",
    "MIR924HG",
    "MIS18BP1",
    "TUBA1C",
    "CDK1",
    "CENPA",
    # extra genes added after LPM subclustering, were not already in SMD gene list
    "KIF11",
    "ECT2",
    "KNL1",
    "NEK2",
    "CEP55",
    "PSRC1",
    "CDCA3",
    "CCNA2",
    "GTSE1",
    "CENPF",
    "CKS1B",
    "MELK",
    "CDCA5",
    # Added 4/8 during LPM sub-subclustering
    "MCM4",
    "HIST1H1E",
    "BRCA2",
    "TRIM66",
    "C1orf112",
    "POLD3",
    "KIF23",
    # Added 4/8 part 2
    "CDKAL1",
    "RAD51AP1",
    "RANBP1",
    "CDCA2",
    "CAND2",
    "PCLAF",
    "EIF5A",
    "SRSF2",
    "GINS1",
    "GINS2",
]

# For LPM / IM / Endothelial subclustering analysis, we repeat preprocessing steps so that
# gene filtering (mean, standard deviation, coefficient of variation) is performed on the
# combined mesoderm/endothelial neighborhood alone before SMD.
# Keep development outputs local to the active workflow lane.
lpm_stage_path = RESULTS_DIR / "intermediates" / "03_lpm_subclustering"
lpm_stage_path.mkdir(parents=True, exist_ok=True)

lpm_smd_input_name = "trunk_morph_lpm_im_endo__smd_input.npz"
lpm_im_endo_clusters = [
    "Lateral Plate Mesoderm",
    "Intermediate Mesoderm",
    "Endothelial",
]

# Historical variable names are retained for continuity with the later notebook cells.
adata_morph_lpm = adata_morph_raw.copy()

# Filter for cells more than 5k reads
lpm_minimum_reads = 5000
sc.pp.filter_cells(adata_morph_lpm, min_counts=lpm_minimum_reads)

# Downsample such that each cell has 10k reads
sc.pp.downsample_counts(
    adata_morph_lpm, counts_per_cell=lpm_minimum_reads, replace=True, random_state=0
)

adata_morph_lpm.obs["leiden_morph"] = adata_morph_with_clusters.obs[
    "leiden_morph"
].reindex(adata_morph_lpm.obs_names)
missing_cluster_labels = int(adata_morph_lpm.obs["leiden_morph"].isna().sum())
if missing_cluster_labels:
    print(
        f"Dropping {missing_cluster_labels} cells not present in adata_morph_with_clusters "
        "before coarse-cluster subsetting."
    )
    adata_morph_lpm = adata_morph_lpm[
        adata_morph_lpm.obs["leiden_morph"].notna()
    ].copy()

# Restrict to the combined LPM / IM / Endothelial neighborhood.
adata_morph_lpm = adata_morph_lpm[
    adata_morph_lpm.obs.leiden_morph.isin(lpm_im_endo_clusters).values
].copy()

subset_cluster_counts = (
    adata_morph_lpm.obs["leiden_morph"].astype(str).value_counts().to_dict()
)
print("Combined 03 pre-SMD subset counts:", subset_cluster_counts)

# Log-transform read counts
sc.pp.log1p(adata_morph_lpm)

# For SMD analysis:
# Filter for log-transformed genes with mean value greater than 0.05 and standard deviation greater than 0.05
adata_morph_lpm_filtered = filter_genes(
    adata_morph_lpm, mean_cutoff=0.05, std_cutoff=0.05
)

# Normalize each gene to have unit variance in expression
adata_morph_lpm_filtered = normalize_unit_variance(adata_morph_lpm_filtered)

# Filter for coefficient of variation (mean / std deviation) > 1.
# Since we have normalized to unit variance this is just a mean cutoff < 1
counts = adata_morph_lpm_filtered.X
cv_mask = np.mean(counts, axis=0).transpose() < 1
adata_morph_lpm_filtered = adata_morph_lpm_filtered[:, cv_mask]

# Save pre-processed expression data to .npz file for SMD on the combined subset.
scipy.sparse.save_npz(
    lpm_stage_path / lpm_smd_input_name,
    adata_morph_lpm_filtered.X,
)

recommended_n_sub = int(round(0.8 * len(adata_morph_lpm_filtered)))
print(
    str(len(adata_morph_lpm_filtered))
    + " cells after pre-processing\n"
    + str(adata_morph_lpm_filtered.shape[1])
    + " genes after preprocessing\nRecommended n_sub ~= "
    + str(recommended_n_sub)
)
print(f"Saved combined SMD input to {lpm_stage_path / lpm_smd_input_name}")

save_json(
    {
        "stage": "03_lpm_subclustering",
        "subset_name": "lpm_im_endo",
        "subset_clusters": lpm_im_endo_clusters,
        "minimum_reads": lpm_minimum_reads,
        "smd_input_file": lpm_smd_input_name,
        "n_cells_after_preprocessing": int(len(adata_morph_lpm_filtered)),
        "n_genes_after_preprocessing": int(adata_morph_lpm_filtered.shape[1]),
        "recommended_n_sub": recommended_n_sub,
        "coarse_cluster_counts_input": subset_cluster_counts,
        "status": "awaiting_new_smd_run",
    },
    lpm_stage_path / "pre_smd_input_meta.json",
)

# For downstream analysis after SMD:
# Only filter genes for non-zero mean and standard deviation:
adata_morph_lpm_nofilter = adata_morph_lpm.copy()
adata_morph_lpm = filter_genes(adata_morph_lpm, mean_cutoff=0, std_cutoff=0)
adata_morph_lpm = normalize_unit_variance(adata_morph_lpm)

## Post-SMD Processing, Clustering, and Output Generation
This block now consumes the local combined `LPM / IM / Endothelial` SMD run, performs
combined mesoderm/endothelial subclustering, and preserves a manual override for the five
reviewed endothelial cells so downstream figures remain stable while we refine objective
recovery later.

In [ ]:
lpm_smd_run_paths = {
    "001": PROJECT_ROOT / "trunk_main_dev" / "results"
    / "smd_runs"
    / "trunk_morph_lpm_run001"
    / "z_trunk_morph_lpm_im_endo__smd_input_001.npy",
    "002": PROJECT_ROOT / "trunk_main_dev" / "results"
    / "smd_runs"
    / "trunk_morph_lpm_run002"
    / "z_trunk_morph_lpm_im_endo__smd_input_002.npy",
}
lpm_smd_run_n_sub = {"001": 396, "002": 396}
lpm_smd_run_trials = {"001": 15000, "002": 60000}
lpm_smd_runs = {run_id: load_npy(path) for run_id, path in lpm_smd_run_paths.items()}
lpm_smd_run_order = sorted(lpm_smd_runs, key=int)
for run_id, z_scores in lpm_smd_runs.items():
    assert len(z_scores) == adata_morph_lpm_filtered.n_vars, (
        run_id,
        len(z_scores),
        adata_morph_lpm_filtered.n_vars,
    )

lpm_smd_run_summary = pd.DataFrame(
    [
        {
            "run_id": run_id,
            "n_sub": lpm_smd_run_n_sub[run_id],
            "trials": lpm_smd_run_trials[run_id],
            "n_genes_z_gt_2": int((lpm_smd_runs[run_id] > 2).sum()),
            "max_z": float(lpm_smd_runs[run_id].max()),
        }
        for run_id in lpm_smd_run_order
    ]
)
display(lpm_smd_run_summary)

selected_lpm_smd_run = "001"
lpm_smd_run_name = f"trunk_morph_lpm_run{selected_lpm_smd_run}"
lpm_smd_zscore_name = f"z_trunk_morph_lpm_im_endo__smd_input_{selected_lpm_smd_run}.npy"
lpm_smd_run_path = lpm_smd_run_paths[selected_lpm_smd_run]
z_scores_morph_lpm = lpm_smd_runs[selected_lpm_smd_run]
adata_morph_lpm_filtered.var['z_score_morph_lpm'] = z_scores_morph_lpm
adata_morph_lpm_filtered.var['log1p_z_score_morph_lpm'] = np.log1p(z_scores_morph_lpm)

comparison_rows = []
selected_gene_set = set(np.flatnonzero(z_scores_morph_lpm > 2))
for run_id in lpm_smd_run_order:
    if run_id == selected_lpm_smd_run:
        continue
    z_scores = lpm_smd_runs[run_id]
    linreg = scipy.stats.linregress(z_scores, z_scores_morph_lpm)
    current_gene_set = set(np.flatnonzero(z_scores > 2))
    union = selected_gene_set | current_gene_set
    comparison_rows.append(
        {
            "reference_run": run_id,
            "selected_run": selected_lpm_smd_run,
            "pearson_r": float(np.corrcoef(z_scores, z_scores_morph_lpm)[0, 1]),
            "slope": float(linreg.slope),
            "intercept": float(linreg.intercept),
            "z_gt_2_jaccard": float(
                len(selected_gene_set & current_gene_set) / len(union)
            ),
            "selected_only_genes": len(selected_gene_set - current_gene_set),
            "reference_only_genes": len(current_gene_set - selected_gene_set),
        }
    )
    with plt.rc_context({"figure.dpi": NOTEBOOK_DPI}):
        plt.figure(figsize=(4, 4))
        plt.plot(
            np.log1p(np.clip(z_scores, 0, None)),
            np.log1p(np.clip(z_scores_morph_lpm, 0, None)),
            "k.",
            markersize=2,
            alpha=0.4,
        )
        plt.xlabel(f"run {run_id} log1p(max(z, 0))")
        plt.ylabel(f"run {selected_lpm_smd_run} log1p(max(z, 0))")
        plt.title(f"Combined LPM SMD: run {run_id} vs run {selected_lpm_smd_run}")
        plt.show()

display(pd.DataFrame(comparison_rows))
print(f"Loaded combined LPM / IM / Endothelial SMD z-scores from {lpm_smd_run_path}")
print(
    f"Using run {selected_lpm_smd_run} for downstream provisional combined LPM / IM / Endothelial clustering."
)

In [ ]:
# Plot top morph_lpm SMD z-scores
smd_cutoff = 10.0

plt.figure(figsize=(12, 4))
plt.hlines(smd_cutoff, -1, 2000, "r")
plt.plot(sorted(z_scores_morph_lpm)[::-1], "k.", markersize=5)
plt.yscale("log")
plt.ylim(0.01, 1.5 * z_scores_morph_lpm.max())
plt.xlim(-1, 1000)
plt.ylabel("combined LPM / IM / Endothelial z-score")
print(f"\n{int((z_scores_morph_lpm > smd_cutoff).sum())} genes with z_score >= cutoff")
plt.show()

comparison_run_ids = [
    run_id for run_id in lpm_smd_run_order if run_id != selected_lpm_smd_run
]


def lpm_run_difference_table(gene_indices):
    rows = []
    for gene_idx in sorted(gene_indices):
        gene_id = str(adata_morph_lpm_filtered.var_names[gene_idx])
        row = {
            "gene_id": gene_id,
            "gene_symbol": adata_morph_lpm_filtered.var.iloc[gene_idx].get(
                "gene_symbol", ""
            ),
            "gene_symbol_original": adata_morph_lpm_filtered.var.iloc[gene_idx].get(
                "gene_symbol_original", ""
            ),
            "selected_z": float(z_scores_morph_lpm[gene_idx]),
        }
        for run_id in comparison_run_ids:
            row[f"run_{run_id}_z"] = float(lpm_smd_runs[run_id][gene_idx])
        rows.append(row)
    if not rows:
        return pd.DataFrame(
            columns=[
                "gene_id",
                "gene_symbol",
                "gene_symbol_original",
                "selected_z",
                *[f"run_{run_id}_z" for run_id in comparison_run_ids],
            ]
        )
    return pd.DataFrame(rows).sort_values("selected_z", ascending=False)


selected_run_gene_set = set(np.flatnonzero(z_scores_morph_lpm > smd_cutoff))
comparison_run_tables = {}
for run_id in comparison_run_ids:
    run_gene_set = set(np.flatnonzero(lpm_smd_runs[run_id] > smd_cutoff))
    selected_only_df = lpm_run_difference_table(selected_run_gene_set - run_gene_set)
    run_only_df = lpm_run_difference_table(run_gene_set - selected_run_gene_set).rename(
        columns={"selected_z": f"run_{selected_lpm_smd_run}_z"}
    )
    if f"run_{run_id}_z" in run_only_df.columns:
        run_only_df = run_only_df.sort_values(f"run_{run_id}_z", ascending=False)
    comparison_run_tables[run_id] = {
        "selected_only": selected_only_df,
        "reference_only": run_only_df,
    }
    print(
        f"Run {selected_lpm_smd_run}-only genes at z > {smd_cutoff:g} relative to run {run_id}: "
        f"{len(selected_only_df)}"
    )
    display(selected_only_df)
    print(
        f"Run {run_id}-only genes at z > {smd_cutoff:g} relative to run {selected_lpm_smd_run}: "
        f"{len(run_only_df)}"
    )
    display(run_only_df)

adata_morph_lpm_filtered_ = adata_morph_lpm_filtered.copy()
adata_morph_lpm_ = adata_morph_lpm.copy()

selected_gene_ids = list(
    adata_morph_lpm_filtered_.var_names[
        adata_morph_lpm_filtered_.var["z_score_morph_lpm"] > smd_cutoff
    ]
)
manual_addgenes = [
    "CDH5",
    "CDH13",
    "PITX1",
    "TBX4",
    "TBX5",
    "HOXA9",
    "SOX2",
    "FLI1",
    "PLVAP",
    "KDR",
    "FLT1",
    "FZD5",
    "RAX",
    "SOX21",
    "ADGRL4",
    "ECSCR",
    "ERG",
    "PTPRB",
    "RMST",
    "DPYSL5",
    "LSAMP",
    "TEK",
    "TIE1",
    "ESAM",
    "PECAM1",
    "RAMP2",
    "LMO2",
    "HES5",
    "NELL2",
    "HAND1",
    "RERG",
    "ALCAM",
    "WT1",
    "CITED1",
    "FST",
]
manual_removegenes = [
    "PAX3",
    "HOXB9",
    "MAP2",
    "ROBO1",
]
manual_add_gene_ids = list(
    dict.fromkeys(
        resolve_symbols(adata_morph_lpm_, manual_addgenes, strict=False, allow_missing=True)
    )
)
manual_remove_gene_ids = list(
    dict.fromkeys(
        resolve_symbols(adata_morph_lpm_, manual_removegenes, strict=False, allow_missing=True)
    )
)
cellcycle_gene_ids = set(
    resolve_symbols(adata_morph_lpm_, genes_cellcycle, strict=False, allow_missing=True)
)
selected_gene_ids = [g for g in selected_gene_ids if g not in cellcycle_gene_ids]
for gene_id in manual_add_gene_ids:
    if gene_id not in selected_gene_ids and gene_id not in cellcycle_gene_ids:
        selected_gene_ids.append(gene_id)
selected_gene_ids = [g for g in selected_gene_ids if g not in manual_remove_gene_ids]
manual_add_gene_symbols_present = [
    symbol for symbol in manual_addgenes if contains_symbol(adata_morph_lpm_, symbol)
]
manual_add_gene_symbols_missing = [
    symbol for symbol in manual_addgenes if not contains_symbol(adata_morph_lpm_, symbol)
]
if manual_add_gene_symbols_present:
    print("Manual LPM add genes present in current data:", manual_add_gene_symbols_present)
if manual_add_gene_symbols_missing:
    print("Manual LPM add genes missing from current data:", manual_add_gene_symbols_missing)

adata_morph_lpm_.var["z_score_morph_lpm"] = np.nan
adata_morph_lpm_.var["log1p_z_score_morph_lpm"] = np.nan
adata_morph_lpm_.var.loc[adata_morph_lpm_filtered_.var_names, "z_score_morph_lpm"] = (
    adata_morph_lpm_filtered_.var["z_score_morph_lpm"]
)
adata_morph_lpm_.var.loc[
    adata_morph_lpm_filtered_.var_names, "log1p_z_score_morph_lpm"
] = adata_morph_lpm_filtered_.var["log1p_z_score_morph_lpm"]

raw_gene_scores = adata_morph_lpm_filtered_.var.copy().reset_index(drop=True)
raw_gene_scores["gene_ids"] = adata_morph_lpm_filtered_.var_names.astype(str)
raw_gene_scores = raw_gene_scores[["gene_ids"] + [c for c in raw_gene_scores.columns if c != "gene_ids"]]
raw_gene_scores["gene_symbol"] = raw_gene_scores["gene_ids"].map(
    adata_morph_lpm_filtered_.var["gene_symbol"].to_dict()
)
raw_gene_scores["gene_symbol_original"] = raw_gene_scores["gene_ids"].map(
    adata_morph_lpm_filtered_.var["gene_symbol_original"].to_dict()
)
raw_gene_scores["selected_at_z_gt_2"] = raw_gene_scores["z_score_morph_lpm"] > 2
raw_gene_scores["selected_at_z_gt_current"] = (
    raw_gene_scores["z_score_morph_lpm"] > smd_cutoff
)
raw_gene_scores["smd_run_name"] = lpm_smd_run_name
raw_gene_scores["smd_run_file"] = lpm_smd_zscore_name
raw_gene_scores["smd_cutoff_current"] = smd_cutoff
save_path = stage_dir(RESULTS_DIR, "03_lpm_subclustering")
save_pickle(raw_gene_scores, save_path / "trunk_morph_lpm_im_endo_smd_gene_scores.pkl")
raw_gene_scores.to_csv(
    save_path / "trunk_morph_lpm_im_endo_smd_gene_scores.csv", index=False
)

all_gene_ids = pd.Index(
    list(
        dict.fromkeys(
            list(adata_morph_lpm_filtered_.var_names)
            + manual_add_gene_ids
            + manual_remove_gene_ids
            + list(cellcycle_gene_ids)
        )
    )
)
final_gene_selection = pd.DataFrame({"gene_ids": all_gene_ids})
final_gene_selection["gene_symbol"] = final_gene_selection["gene_ids"].map(
    adata_morph_lpm_.var["gene_symbol"].to_dict()
)
final_gene_selection["gene_symbol_original"] = final_gene_selection["gene_ids"].map(
    adata_morph_lpm_.var["gene_symbol_original"].to_dict()
)
final_gene_selection["in_smd_scored_space"] = final_gene_selection["gene_ids"].isin(
    adata_morph_lpm_filtered_.var_names
)
final_gene_selection["z_score_morph_lpm"] = final_gene_selection["gene_ids"].map(
    adata_morph_lpm_.var["z_score_morph_lpm"].to_dict()
)
final_gene_selection["log1p_z_score_morph_lpm"] = final_gene_selection["gene_ids"].map(
    adata_morph_lpm_.var["log1p_z_score_morph_lpm"].to_dict()
)
final_gene_selection["selected_at_z_gt_2"] = (
    final_gene_selection["z_score_morph_lpm"] > 2
)
final_gene_selection["selected_at_z_gt_current"] = (
    final_gene_selection["z_score_morph_lpm"] > smd_cutoff
)
final_gene_selection["manually_added_to_final_smd"] = final_gene_selection["gene_ids"].isin(
    manual_add_gene_ids
)
final_gene_selection["manually_removed_from_final_smd"] = final_gene_selection["gene_ids"].isin(
    manual_remove_gene_ids
)
final_gene_selection["removed_as_cell_cycle"] = final_gene_selection["gene_ids"].isin(
    cellcycle_gene_ids
)
final_gene_selection["included_in_final_smd_current"] = (
    (
        final_gene_selection["selected_at_z_gt_current"]
        | final_gene_selection["manually_added_to_final_smd"]
    )
    & ~final_gene_selection["manually_removed_from_final_smd"]
    & ~final_gene_selection["removed_as_cell_cycle"]
)
save_pickle(
    final_gene_selection, save_path / "trunk_morph_lpm_im_endo_final_gene_selection.pkl"
)
final_gene_selection.to_csv(
    save_path / "trunk_morph_lpm_im_endo_final_gene_selection.csv", index=False
)

adata_morph_lpm_SMD = adata_morph_lpm_[:, selected_gene_ids].copy()
print(
    f"{adata_morph_lpm_SMD.n_vars} genes selected with z > {smd_cutoff:g} after cell-cycle removal"
)

kept_smd_gene_table = (
    final_gene_selection.loc[final_gene_selection["included_in_final_smd_current"]]
    .loc[
        :,
        [
            "gene_ids",
            "gene_symbol",
            "gene_symbol_original",
            "z_score_morph_lpm",
            "log1p_z_score_morph_lpm",
            "manually_added_to_final_smd",
        ],
    ]
    .sort_values("z_score_morph_lpm", ascending=False)
    .reset_index(drop=True)
)
with pd.option_context(
    "display.max_rows",
    max(len(kept_smd_gene_table), 200),
    "display.min_rows",
    max(len(kept_smd_gene_table), 200),
    "display.max_columns",
    None,
):
    display(kept_smd_gene_table)

try:
    kept_smd_gene_table
except Exception as err:
    print(f"Skipping clipboard export for LPM SMD z-scores: {err}")

adata_morph_lpm_SMD_ = adata_morph_lpm_SMD.copy()
sc.pp.normalize_total(adata_morph_lpm_SMD_)
sc.pp.neighbors(adata_morph_lpm_SMD_, use_rep="X", method="gauss")
try:
    sc.tl.umap(adata_morph_lpm_SMD_, random_state=0)
except Exception as err:
    print(f"Skipping LPM UMAP computation: {err}")

precluster_stage_path = stage_dir(RESULTS_DIR, "03_lpm_subclustering")
for _adata in [adata_morph_lpm_, adata_morph_lpm_SMD_, adata_morph_lpm_SMD]:
    assert_gene_id_index(_adata)
save_h5ad(adata_morph_lpm_, precluster_stage_path / "adata_morph_lpmendo_.h5ad")
save_h5ad(adata_morph_lpm_SMD_, precluster_stage_path / "adata_morph_lpmendo_SMD_.h5ad")
save_h5ad(adata_morph_lpm_SMD, precluster_stage_path / "adata_morph_lpmendo_SMD.h5ad")
save_json(
    {
        "stage": "03_lpm_subclustering",
        "status": "precluster_substrate_exported",
        "subset_clusters": lpm_im_endo_clusters,
        "smd_run_name": lpm_smd_run_name,
        "smd_run_file": lpm_smd_zscore_name,
        "smd_cutoff_current": smd_cutoff,
        "manual_add_genes": manual_addgenes,
        "manual_remove_genes": manual_removegenes,
        "n_cells_precluster": int(adata_morph_lpm_SMD_.n_obs),
        "n_genes_precluster": int(adata_morph_lpm_SMD_.n_vars),
        "gene_selection_rule": "z_gt_cutoff_only_then_remove_cell_cycle",
    },
    precluster_stage_path / "meta.json",
)
print(f"Saved precluster LPM / IM / Endothelial substrates to {precluster_stage_path}")

print("Prepared stage-03 precluster substrates; applying final leiden_ensemble-backed clustering settings below.")


In [ ]:

FINAL_LPM_LEIDEN_RESOLUTION = 1.1
FINAL_LPM_LEIDEN_RANDOM_STATE = 5
FINAL_LPM_SELECTION_RULE = "medoid_raw_run_by_mean_ari"
FINAL_LPM_SELECTION_SOURCE = "leiden_ensemble 03_lpm_subclustering representative run"

raw_cluster_map = {
    "0": "Posterior Trunk LPM",
    "1": "Neuroectodermal Contamination",
    "2": "Endothelial",
    "3": "Anterior Trunk LPM",
    "4": "Intermediate Mesoderm",
}
final_lpm_celltype_order = [
    "Endothelial",
    "Anterior Trunk LPM",
    "Posterior Trunk LPM",
    "Intermediate Mesoderm",
    "Neuroectodermal Contamination",
]

sc.tl.leiden(
    adata_morph_lpm_SMD_,
    resolution=FINAL_LPM_LEIDEN_RESOLUTION,
    random_state=FINAL_LPM_LEIDEN_RANDOM_STATE,
    key_added="leiden_morph_lpm_raw",
    flavor="igraph",
    n_iterations=-1,
)
adata_morph_lpm_SMD_.obs["leiden_morph_lpm_raw"] = (
    adata_morph_lpm_SMD_.obs["leiden_morph_lpm_raw"].astype(str)
)

adata_morph_lpm_SMD_.obs["leiden_morph_lpm"] = (
    adata_morph_lpm_SMD_.obs["leiden_morph_lpm_raw"]
    .map(raw_cluster_map)
    .astype(pd.CategoricalDtype(categories=final_lpm_celltype_order, ordered=True))
)
adata_morph_lpm_SMD.obs["leiden_morph_lpm_raw"] = adata_morph_lpm_SMD_.obs[
    "leiden_morph_lpm_raw"
].reindex(adata_morph_lpm_SMD.obs_names)
adata_morph_lpm_SMD.obs["leiden_morph_lpm"] = adata_morph_lpm_SMD_.obs[
    "leiden_morph_lpm"
].reindex(adata_morph_lpm_SMD.obs_names)
adata_morph_lpm_.obs["leiden_morph_lpm_raw"] = adata_morph_lpm_SMD_.obs[
    "leiden_morph_lpm_raw"
].reindex(adata_morph_lpm_.obs_names)
adata_morph_lpm_.obs["leiden_morph_lpm"] = adata_morph_lpm_SMD_.obs[
    "leiden_morph_lpm"
].reindex(adata_morph_lpm_.obs_names)

print("Final combined LPM / IM / Endothelial raw cluster counts at resolution 1.1, seed 5:")
display(adata_morph_lpm_SMD_.obs["leiden_morph_lpm_raw"].value_counts().sort_index())
print("Final combined LPM / IM / Endothelial label counts:")
display(adata_morph_lpm_SMD_.obs["leiden_morph_lpm"].value_counts())


### Visual Diagnostics and Figure Exports
Restore the parity-style inspection plots for the combined `LPM / IM / Endothelial` neighborhood while preserving the current dev clustering, ensemble-selected seed/resolution, and broadened stage scope.


In [ ]:
qc_dir = RESULTS_DIR / "qc" / "03_lpm_subclustering"
qc_dir.mkdir(parents=True, exist_ok=True)

# Gene-gene correlation among the final selected combined-neighborhood features.
corr_morph_lpm_gg = gene_corrcoef_sparse_safe(adata_morph_lpm_SMD.X)
with plt.rc_context({"figure.dpi": NOTEBOOK_DPI}):
    cluster_gg_morph_lpm = sns.clustermap(
        corr_morph_lpm_gg,
        method="ward",
        metric="euclidean",
        figsize=(16, 16),
        cmap=batlow,
        vmin=-0.2,
        vmax=0.5,
        yticklabels=var_names_to_symbols(adata_morph_lpm_SMD),
        xticklabels=var_names_to_symbols(adata_morph_lpm_SMD),
    )
    cluster_gg_morph_lpm.fig.suptitle(
        "Combined LPM / IM / Endothelial gene-gene correlation", y=1.02
    )
    cluster_gg_morph_lpm.fig.savefig(
        qc_dir / "lpm_im_endo_gene_gene_correlation_clustermap.png",
        bbox_inches="tight",
    )
    plt.show()

# Cluster-cluster mean-expression correlation across the final combined-neighborhood labels.
adata_morph_lpm_SMD_leiden = sc.get.aggregate(
    adata_morph_lpm_SMD_,
    by="leiden_morph_lpm",
    func=["count_nonzero", "mean", "sum", "var"],
    axis="obs",
)
corr_clcl_morph_lpm = np.corrcoef(adata_morph_lpm_SMD_leiden.layers["mean"])
with plt.rc_context({"figure.figsize": (6, 5), "figure.dpi": NOTEBOOK_DPI}):
    cluster_clcl_morph_lpm = sns.clustermap(
        corr_clcl_morph_lpm,
        method="ward",
        metric="euclidean",
        figsize=(8, 8),
        cmap=batlow,
        yticklabels=adata_morph_lpm_SMD_leiden.obs.leiden_morph_lpm.cat.categories,
        xticklabels=adata_morph_lpm_SMD_leiden.obs.leiden_morph_lpm.cat.categories,
    )
    cluster_clcl_morph_lpm.fig.suptitle(
        "Cluster-cluster mean gene correlation", y=1.02
    )
    cluster_clcl_morph_lpm.fig.savefig(
        qc_dir / "lpm_im_endo_cluster_cluster_correlation.png",
        bbox_inches="tight",
    )
    plt.show()

# UMAP and source-composition plots for the final combined-neighborhood partition.
with plt.rc_context({"figure.figsize": (6, 5), "figure.dpi": 300}):
    sc.pl.umap(
        adata_morph_lpm_SMD_,
        color="leiden_morph_lpm",
        size=40,
        title="Day 6 Bead + No Bead Morph LPM / IM / Endothelial Cells",
        show=False,
    )
    plt.savefig(
        qc_dir / "lpm_im_endo_umap_final_clusters.png",
        bbox_inches="tight",
        pad_inches=0,
    )
    plt.show()

import matplotlib.ticker as mtick

with plt.rc_context({"figure.figsize": (7, 4), "figure.dpi": 300}):
    source_tab = pd.crosstab(
        adata_morph_lpm_SMD_.obs["source"],
        adata_morph_lpm_SMD_.obs["leiden_morph_lpm"],
        normalize="columns",
    ).T
    ax = source_tab.plot(kind="bar", stacked=True)
    ax.set_xlabel("Final combined neighborhood label")
    ax.set_ylabel("Fraction of cells")
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    ax.legend(title="Source", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.savefig(
        qc_dir / "lpm_im_endo_cluster_composition_by_source.png",
        bbox_inches="tight",
    )
    plt.show()

# Broad all-cluster diagnostic heatmap for the combined neighborhood.
diagnostic_gene_panel = [
    # Endothelial
    "CDH5",
    "PECAM1",
    "KDR",
    "FLT1",
    "ESAM",
    "ERG",
    "RAMP2",
    # Anterior trunk LPM
    "TBX5",
    "HAND1",
    "FLRT2",
    "RYR2",
    "KCNQ5",
    "RELN",
    "TNNI1",
    "BAMBI",
    # Posterior trunk LPM
    "PITX1",
    "HOXA9",
    "HOXB9",
    "HOXC9",
    "EYA2",
    "TRPS1",
    "DST",
    "PDZD2",
    # Intermediate mesoderm
    "WT1",
    "CITED1",
    "FST",
    # Neuroectodermal contamination
    "SOX2",
    "SOX21",
    "RAX",
    "HES5",
    "RMST",
    "NELL2",
    "DPYSL5",
]
missing_diagnostic_symbols = symbols_missing(adata_morph_lpm_, diagnostic_gene_panel)
if missing_diagnostic_symbols:
    print("Diagnostic combined-neighborhood heatmap genes missing from current data:", missing_diagnostic_symbols)

diagnostic_gene_ids = resolve_symbols(
    adata_morph_lpm_, diagnostic_gene_panel, strict=False, allow_missing=True
)
adata_morph_lpm_diag = adata_morph_lpm_[:, diagnostic_gene_ids].copy()
adata_morph_lpm_diag = adata_morph_lpm_diag[
    adata_morph_lpm_diag.obs["leiden_morph_lpm"].notna()
].copy()
diagnostic_order = ordered_cells_by_cluster_clustermap(
    adata=adata_morph_lpm_diag,
    cluster_key="leiden_morph_lpm",
    var_names=adata_morph_lpm_diag.var_names,
    figsize=(10, 10),
    vmin=0,
    vmax=5,
    cmap=batlow,
    method="ward",
    metric="euclidean",
)
adata_morph_lpm_diag_sorted = adata_morph_lpm_diag[diagnostic_order, :].copy()
with plt.rc_context({"figure.dpi": 300}):
    ax = sc.pl.heatmap(
        adata_morph_lpm_diag_sorted,
        var_names=adata_morph_lpm_diag_sorted.var_names,
        groupby="leiden_morph_lpm",
        swap_axes=True,
        show_gene_labels=True,
        figsize=(11, 11),
        vmin=0,
        vmax=5,
        cmap=batlow,
        show=False,
    )
    plt.savefig(
        qc_dir / "lpm_im_endo_diagnostic_heatmap_all_clusters.png",
        bbox_inches="tight",
        pad_inches=0,
    )
    plt.show()

# Regenerate the manuscript-facing Fig. 4c heatmap assets for the bead-exposed subset.
fig4c_gene_panel = [
    # Endothelial
    "CD34",
    "GNG11",
    "CDH5",
    "HOPX",
    "FLI1",
    "CDH13",
    # Anterior trunk LPM
    "FLRT2",
    "KCNQ5",
    "RELN",
    "RYR2",
    "BAMBI",
    "TNNI1",
    "TNNT2",
    "LINC01099",
    "KCNJ3",
    "ZNF804A",
    "TBX5",
    "NR2F2",
    # Posterior trunk LPM
    "DST",
    "PDZD2",
    "PITX1",
    "HOXB9",
    "HOXA9",
    "HOXC9",
    "HOXA10",
    "EYA2",
]
fig4c_celltype_order = [
    "Endothelial",
    "Anterior Trunk LPM",
    "Posterior Trunk LPM",
]
missing_fig4c_symbols = symbols_missing(adata_morph_lpm_SMD_, fig4c_gene_panel)
if missing_fig4c_symbols:
    print("Fig. 4c genes missing from current dev stage-03 data:", missing_fig4c_symbols)
fig4c_gene_ids = resolve_symbols(
    adata_morph_lpm_SMD_, fig4c_gene_panel, strict=False, allow_missing=True
)
adata_morph_lpm_fig4c = adata_morph_lpm_SMD_[
    (
        adata_morph_lpm_SMD_.obs["source"] == "BMP4 Bead Morph"
    )
    & adata_morph_lpm_SMD_.obs["leiden_morph_lpm"].isin(fig4c_celltype_order),
    fig4c_gene_ids,
].copy()
fig4c_series = adata_morph_lpm_fig4c.obs["leiden_morph_lpm"].cat.remove_unused_categories()
adata_morph_lpm_fig4c.obs["leiden_morph_lpm"] = fig4c_series.cat.reorder_categories(
    [ct for ct in fig4c_celltype_order if ct in fig4c_series.cat.categories]
)
fig4c_cell_order = ordered_cells_by_cluster_clustermap(
    adata=adata_morph_lpm_fig4c,
    cluster_key="leiden_morph_lpm",
    var_names=adata_morph_lpm_fig4c.var_names,
    figsize=(10, 10),
    vmin=0,
    vmax=5,
    cmap=batlow,
    method="ward",
    metric="euclidean",
)
adata_morph_lpm_fig4c_sorted = adata_morph_lpm_fig4c[fig4c_cell_order, :].copy()
fig4c_highlight_symbols = [
    "TBX5", "HOXB3", "HOXB4", "GATA6", "FLRT2", "KCNQ5", "RELN", "TNNT2", "TNNI1", "RYR2", "BAMBI",
    "PITX1", "HOXB9", "HOXA9", "HOXC9", "HOXA10", "TWIST1",
    "TFPI", "CDH5", "CD34", "FLI1", "GNG11", "CDH13",
]
with plt.rc_context({"figure.dpi": 300}):
    ax = sc.pl.heatmap(
        adata_morph_lpm_fig4c_sorted,
        var_names=adata_morph_lpm_fig4c_sorted.var_names,
        groupby="leiden_morph_lpm",
        swap_axes=True,
        show_gene_labels=True,
        figsize=(10, 10),
        vmin=0,
        vmax=5,
        cmap=batlow,
        show=False,
    )
    bold_selected_heatmap_yticklabels(ax, fig4c_highlight_symbols)
    plt.savefig(
        MANUSCRIPT_FIG_DIR / "Fig4c_heatmap_LPM_key_genes_bead.png",
        bbox_inches="tight",
        pad_inches=0,
    )
    plt.show()

with plt.rc_context({"figure.dpi": 300}):
    fig_dict = sc.pl.heatmap(
        adata_morph_lpm_fig4c_sorted,
        var_names=adata_morph_lpm_fig4c_sorted.var_names,
        groupby="leiden_morph_lpm",
        swap_axes=True,
        figsize=(10, 10),
        vmin=0,
        vmax=5,
        cmap=batlow,
        show=False,
        var_group_labels=[],
    )
    hide_heatmap_axes_and_colorbar(fig_dict)
    plt.savefig(
        MANUSCRIPT_FIG_DIR / "Fig4c_heatmap_LPM_key_genes_bead_noaxes.png",
        bbox_inches="tight",
        pad_inches=0,
    )
    plt.show()

with plt.rc_context({"figure.dpi": 300}):
    ax = plot_empty_heatmap_axes(
        adata_morph_lpm_fig4c_sorted,
        var_names=adata_morph_lpm_fig4c_sorted.var_names,
        groupby="leiden_morph_lpm",
        swap_axes=True,
        show_gene_labels=True,
        figsize=(10, 10),
        vmin=0,
        vmax=5,
        cmap=batlow,
        show=False,
    )
    keep_only_selected_heatmap_yticklabels(ax, fig4c_highlight_symbols)
    plt.savefig(
        MANUSCRIPT_FIG_DIR / "Fig4c_heatmap_LPM_key_genes_bead_axesonly.svg",
        bbox_inches="tight",
        pad_inches=0,
    )
    plt.show()

# Alternate Fig. 4c heatmap using the broader stage-03 gene space and the updated
# combined LPM / IM / Endothelial annotation set. Keep the original Fig. 4c assets
# untouched and export this as a parallel figure family for comparison.
fig4c_alt_gene_panel = [
    # Endothelial
    "CDH5",
    "PLVAP",
    "GNG11",
    "ERG",
    # Anterior trunk LPM
    "HAND1",
    "FLRT2",
    "RYR2",
    "BAMBI",
    "NPNT",
    # Posterior trunk LPM
    "PITX1",
    "TRPS1",
    "PDZD2",
    "EPHA3",
    # Shared posterior trunk program across posterior LPM and IM
    "HOXA9",
    "HOXB9",
    "HOXC9",
    # Intermediate mesoderm
    "PAX8",
    "EYA1",
    "CDH6",
    "GREM1",
    "FST",
]
fig4c_alt_celltype_order = [
    "Endothelial",
    "Anterior Trunk LPM",
    "Posterior Trunk LPM",
    "Intermediate Mesoderm",
]
missing_fig4c_alt_symbols = symbols_missing(adata_morph_lpm_, fig4c_alt_gene_panel)
if missing_fig4c_alt_symbols:
    print(
        "Alternate Fig. 4c genes missing from current dev stage-03 broad data:",
        missing_fig4c_alt_symbols,
    )
fig4c_alt_gene_ids = resolve_symbols(
    adata_morph_lpm_, fig4c_alt_gene_panel, strict=False, allow_missing=True
)
adata_morph_lpm_fig4c_alt = adata_morph_lpm_[
    (
        adata_morph_lpm_.obs["source"] == "BMP4 Bead Morph"
    )
    & adata_morph_lpm_.obs["leiden_morph_lpm"].isin(fig4c_alt_celltype_order),
    fig4c_alt_gene_ids,
].copy()
fig4c_alt_series = adata_morph_lpm_fig4c_alt.obs["leiden_morph_lpm"].cat.remove_unused_categories()
adata_morph_lpm_fig4c_alt.obs["leiden_morph_lpm"] = (
    fig4c_alt_series.cat.reorder_categories(
        [ct for ct in fig4c_alt_celltype_order if ct in fig4c_alt_series.cat.categories]
    )
)
fig4c_alt_cell_order = ordered_cells_by_cluster_clustermap(
    adata=adata_morph_lpm_fig4c_alt,
    cluster_key="leiden_morph_lpm",
    var_names=adata_morph_lpm_fig4c_alt.var_names,
    figsize=(10, 10),
    vmin=0,
    vmax=5,
    cmap=batlow,
    method="ward",
    metric="euclidean",
)
adata_morph_lpm_fig4c_alt_sorted = adata_morph_lpm_fig4c_alt[
    fig4c_alt_cell_order, :
].copy()
fig4c_alt_highlight_symbols = fig4c_alt_gene_panel.copy()
with plt.rc_context({"figure.dpi": 300}):
    ax = sc.pl.heatmap(
        adata_morph_lpm_fig4c_alt_sorted,
        var_names=adata_morph_lpm_fig4c_alt_sorted.var_names,
        groupby="leiden_morph_lpm",
        swap_axes=True,
        show_gene_labels=True,
        figsize=(10, 10),
        vmin=0,
        vmax=5,
        cmap=batlow,
        show=False,
    )
    bold_selected_heatmap_yticklabels(ax, fig4c_alt_highlight_symbols)
    plt.savefig(
        MANUSCRIPT_FIG_DIR / "Fig4c_heatmap_LPM_key_genes_bead_alt_im.png",
        bbox_inches="tight",
        pad_inches=0,
    )
    plt.show()

with plt.rc_context({"figure.dpi": 300}):
    fig_dict = sc.pl.heatmap(
        adata_morph_lpm_fig4c_alt_sorted,
        var_names=adata_morph_lpm_fig4c_alt_sorted.var_names,
        groupby="leiden_morph_lpm",
        swap_axes=True,
        figsize=(10, 10),
        vmin=0,
        vmax=5,
        cmap=batlow,
        show=False,
        var_group_labels=[],
    )
    hide_heatmap_axes_and_colorbar(fig_dict)
    plt.savefig(
        MANUSCRIPT_FIG_DIR / "Fig4c_heatmap_LPM_key_genes_bead_alt_im_noaxes.png",
        bbox_inches="tight",
        pad_inches=0,
    )
    plt.show()

with plt.rc_context({"figure.dpi": 300}):
    ax = plot_empty_heatmap_axes(
        adata_morph_lpm_fig4c_alt_sorted,
        var_names=adata_morph_lpm_fig4c_alt_sorted.var_names,
        groupby="leiden_morph_lpm",
        swap_axes=True,
        show_gene_labels=True,
        figsize=(10, 10),
        vmin=0,
        vmax=5,
        cmap=batlow,
        show=False,
    )
    keep_only_selected_heatmap_yticklabels(ax, fig4c_alt_highlight_symbols)
    plt.savefig(
        MANUSCRIPT_FIG_DIR / "Fig4c_heatmap_LPM_key_genes_bead_alt_im_axesonly.svg",
        bbox_inches="tight",
        pad_inches=0,
    )
    plt.show()

print(f"Saved stage-03 diagnostic plots to {qc_dir}")
print(f"Regenerated Fig. 4c manuscript heatmap assets in {MANUSCRIPT_FIG_DIR}")


### Final Fig. 4c With IM Export
Export the finalized `Fig4c` variant that preserves the parity-style panel logic, appends `Intermediate Mesoderm`, and writes full/data-only/axes-only assets in the same style as the stage-02 heatmaps.


In [ ]:

def add_fate_boundary_separators(ax_dict, ordered_obs, width_cells=1.0):
    ordered_strings = ordered_obs.astype(str)
    if hasattr(ordered_obs.dtype, "categories"):
        categories = [
            c for c in ordered_obs.cat.categories if c in set(ordered_strings)
        ]
    else:
        categories = list(dict.fromkeys(ordered_strings))

    counts = [int((ordered_strings == str(cat)).sum()) for cat in categories]
    boundaries = np.cumsum(counts)[:-1]
    for boundary in boundaries:
        center = boundary - 0.5
        x0 = center - width_cells / 2
        x1 = center + width_cells / 2
        ax_dict["heatmap_ax"].axvspan(x0, x1, color="white", zorder=20)
        if "groupby_ax" in ax_dict and ax_dict["groupby_ax"] is not None:
            ax_dict["groupby_ax"].axvspan(x0, x1, color="white", zorder=20)


fig4c_with_im_gene_panel = [
    "CD34",
    "GNG11",
    "CDH5",
    "ERG",
    "FLI1",
    "CDH13",
    "PRRX1",
    "HAND1",
    "FLRT2",
    "KCNQ5",
    "RELN",
    "RYR2",
    "BAMBI",
    "TNNI1",
    "DST",
    "PDZD2",
    "PITX1",
    "NR2F2",
    "HOXB9",
    "HOXA9",
    "EYA1",
    "SIX1",
    "GREM1",
    "FST",
    "ITGA8",
    "PAX8",
    "CDH6",
    "SALL1",
    "WT1",
    "OSR1",
    "CITED1",
]
fig4c_with_im_bold_symbols = [
    "CD34",
    "GNG11",
    "CDH5",
    "CDH13",
    "PRRX1",
    "HAND1",
    "FLRT2",
    "KCNQ5",
    "RYR2",
    "TNNI1",
    "PITX1",
    "HOXB9",
    "HOXA9",
    "EYA1",
    "ITGA8",
    "PAX8",
    "CDH6",
    "WT1",
]
fig4c_with_im_celltype_order = [
    "Endothelial",
    "Anterior Trunk LPM",
    "Posterior Trunk LPM",
    "Intermediate Mesoderm",
]

missing_fig4c_with_im_symbols = symbols_missing(adata_morph_lpm_, fig4c_with_im_gene_panel)
if missing_fig4c_with_im_symbols:
    print(
        "Final Fig. 4c with IM genes missing from current dev stage-03 broad data:",
        missing_fig4c_with_im_symbols,
    )

fig4c_with_im_gene_ids = resolve_symbols(
    adata_morph_lpm_, fig4c_with_im_gene_panel, strict=False, allow_missing=True
)
adata_morph_lpm_fig4c_with_im = adata_morph_lpm_[
    (
        adata_morph_lpm_.obs["source"] == "BMP4 Bead Morph"
    )
    & adata_morph_lpm_.obs["leiden_morph_lpm"].isin(fig4c_with_im_celltype_order),
    fig4c_with_im_gene_ids,
].copy()
fig4c_with_im_series = (
    adata_morph_lpm_fig4c_with_im.obs["leiden_morph_lpm"]
    .cat.remove_unused_categories()
)
adata_morph_lpm_fig4c_with_im.obs["leiden_morph_lpm"] = (
    fig4c_with_im_series.cat.reorder_categories(
        [ct for ct in fig4c_with_im_celltype_order if ct in fig4c_with_im_series.cat.categories]
    )
)
fig4c_with_im_cell_order = ordered_cells_by_cluster_clustermap(
    adata=adata_morph_lpm_fig4c_with_im,
    cluster_key="leiden_morph_lpm",
    var_names=adata_morph_lpm_fig4c_with_im.var_names,
    figsize=(10, 10),
    vmin=0,
    vmax=5,
    cmap=batlow,
    method="ward",
    metric="euclidean",
)
adata_morph_lpm_fig4c_with_im_sorted = adata_morph_lpm_fig4c_with_im[
    fig4c_with_im_cell_order, :
].copy()

fig4c_with_im_stem = "Fig4c_heatmap_LPM_key_genes_bead_with_IM"

with plt.rc_context({"figure.dpi": EXPORT_DPI}):
    ax = sc.pl.heatmap(
        adata_morph_lpm_fig4c_with_im_sorted,
        var_names=adata_morph_lpm_fig4c_with_im_sorted.var_names,
        groupby="leiden_morph_lpm",
        swap_axes=True,
        show_gene_labels=True,
        figsize=(10, 12),
        vmin=0,
        vmax=5,
        cmap=batlow,
        show=False,
    )
    add_fate_boundary_separators(
        ax, adata_morph_lpm_fig4c_with_im_sorted.obs["leiden_morph_lpm"]
    )
    bold_selected_heatmap_yticklabels(ax, fig4c_with_im_bold_symbols)
    for ext in ["png", "pdf", "svg"]:
        plt.savefig(
            MANUSCRIPT_FIG_DIR / f"{fig4c_with_im_stem}.{ext}",
            bbox_inches="tight",
            pad_inches=0,
            dpi=300,
        )
    plt.show()

with plt.rc_context({"figure.dpi": EXPORT_DPI}):
    fig_dict = sc.pl.heatmap(
        adata_morph_lpm_fig4c_with_im_sorted,
        var_names=adata_morph_lpm_fig4c_with_im_sorted.var_names,
        groupby="leiden_morph_lpm",
        swap_axes=True,
        figsize=(10, 12),
        vmin=0,
        vmax=5,
        cmap=batlow,
        show=False,
        var_group_labels=[],
    )
    add_fate_boundary_separators(
        fig_dict, adata_morph_lpm_fig4c_with_im_sorted.obs["leiden_morph_lpm"]
    )
    hide_heatmap_axes_and_colorbar(fig_dict)
    plt.savefig(
        MANUSCRIPT_FIG_DIR / f"{fig4c_with_im_stem}_noaxes.png",
        bbox_inches="tight",
        pad_inches=0,
        dpi=300,
    )
    plt.show()

with plt.rc_context({"figure.dpi": EXPORT_DPI}):
    ax = plot_empty_heatmap_axes(
        adata_morph_lpm_fig4c_with_im_sorted,
        var_names=adata_morph_lpm_fig4c_with_im_sorted.var_names,
        groupby="leiden_morph_lpm",
        swap_axes=True,
        show_gene_labels=True,
        figsize=(10, 12),
        vmin=0,
        vmax=5,
        cmap=batlow,
        show=False,
    )
    add_fate_boundary_separators(
        ax, adata_morph_lpm_fig4c_with_im_sorted.obs["leiden_morph_lpm"]
    )
    keep_only_selected_heatmap_yticklabels(ax, fig4c_with_im_bold_symbols)
    bold_selected_heatmap_yticklabels(ax, fig4c_with_im_bold_symbols)
    ax["groupby_ax"].set_xlabel("")
    for ext in ["svg", "pdf"]:
        plt.savefig(
            MANUSCRIPT_FIG_DIR / f"{fig4c_with_im_stem}_axesonly.{ext}",
            bbox_inches="tight",
            pad_inches=0,
            dpi=300,
        )
    plt.show()


## Save Stage Outputs
Persist LPM subclustering intermediates and metadata for downstream notebooks.


In [ ]:

stage_path = stage_dir(RESULTS_DIR, "03_lpm_subclustering")
for _adata in [adata_morph_lpm_, adata_morph_lpm_SMD, adata_morph_lpm_SMD_]:
    assert_gene_id_index(_adata)
save_h5ad(adata_morph_lpm_, stage_path / "adata_morph_lpmendo_.h5ad")
save_h5ad(adata_morph_lpm_SMD, stage_path / "adata_morph_lpmendo_SMD.h5ad")
save_h5ad(adata_morph_lpm_SMD_, stage_path / "adata_morph_lpmendo_SMD_.h5ad")

save_json(
    {
        "stage": "03_lpm_subclustering",
        "status": "clustered",
        "subset_clusters": lpm_im_endo_clusters,
        "smd_run_name": lpm_smd_run_name,
        "smd_run_file": lpm_smd_zscore_name,
        "selected_lpm_smd_run": selected_lpm_smd_run,
        "selected_lpm_smd_run_n_sub": lpm_smd_run_n_sub[selected_lpm_smd_run],
        "selected_lpm_smd_run_trials": lpm_smd_run_trials[selected_lpm_smd_run],
        "available_lpm_smd_runs": lpm_smd_run_order,
        "smd_cutoff_current": smd_cutoff,
        "manual_add_genes": manual_addgenes,
        "manual_remove_genes": manual_removegenes,
        "gene_selection_rule": "z_gt_cutoff_only_then_remove_cell_cycle",
        "leiden_resolution": FINAL_LPM_LEIDEN_RESOLUTION,
        "leiden_random_state": FINAL_LPM_LEIDEN_RANDOM_STATE,
        "selection_rule": FINAL_LPM_SELECTION_RULE,
        "selection_source": FINAL_LPM_SELECTION_SOURCE,
        "raw_cluster_map": raw_cluster_map,
        "celltype_order": final_lpm_celltype_order,
        "n_cells_precluster": int(adata_morph_lpm_SMD_.n_obs),
        "n_genes_precluster": int(adata_morph_lpm_SMD_.n_vars),
    },
    stage_path / "meta.json",
)
print(f"Saved final combined LPM / IM / Endothelial intermediates to {stage_path}")
